In [ ]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0

In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
import shutil

shutil.move("research2.pdf", "data/research2.pdf")
shutil.move("bookchapter.pdf", "data/bookchapter.pdf")
shutil.move("Python.txt", "data/Python.txt")

'data/Python.txt'

In [ ]:
from langchain_core.documents import Document

In [ ]:
sample_doc = Document(
    page_content = "Hello World!",
    metadata = {"source": "https://www.google.com"}
)

In [ ]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [ ]:
type(sample_doc)

langchain_core.documents.base.Document

In [ ]:
# # Text Data
# from langchain_community.document_loaders.text import TextLoader
# loader = TextLoader("data/Python.txt", encoding="latin-1")

In [ ]:
# document = loader.load()

In [ ]:
# document

In [ ]:
# # PDF Data
# from langchain_community.document_loaders.pdf import PyPDFLoader
# pdf_loader = PyPDFLoader("data/research2.pdf")
# document = pdf_loader.load()
# document

Ingestion Pipeline

In [ ]:
# Data->Document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [ ]:
def load_all_pdfs():
  folder_path = "data"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      pdf_path = os.path.join(folder_path, filename)

      try:
        loader = PyPDFLoader(pdf_path)
        doc = loader.load()

        all_docs.extend(doc)
        num_docs += 1
      except Exception as e:
        print(f"Error loading PDF {filename}: {e}")

  print("total_pdfs:", num_docs)
  print("total_pages:", len(all_docs))
  return all_docs

In [ ]:
all_pdf_documents = load_all_pdfs()

total_pdfs: 2
total_pages: 48


In [ ]:
# Chunks
!pip install langchain_text_splitters


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split_doc(documents, chunk_size=500, chunk_overlap=50):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )
  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [ ]:
chunks = split_doc(all_pdf_documents)

In [ ]:
len(chunks)

390

Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
  def __init__(self, model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("Loading model", self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimension=", self.model.get_sentence_embedding_dimension())
  def generate_embeddings(self, text):
    embeddings = self.model.encode(text)
    print("embedding shape:", embeddings.shape)
    return embeddings

In [ ]:
embedding_manager = EmbeddingManager()

Loading model all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimension= 384


/tmp/ipykernel_2928/4022114334.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimension=", self.model.get_sentence_embedding_dimension())


Vector Store

In [ ]:
import chromadb
import uuid

In [ ]:
class VectorStoreManager:
  def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None

    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory, exist_ok=True)
    #create a client
    self.client = chromadb.PersistentClient(path=self.persist_directory)
    #create the collection
    self.collection = self.client.get_or_create_collection(
      name=self.collection_name,
      metadata={"hnsw:space": "cosine"}  # ← add this
    )

    print("initialize the vector store with collection", self.collection_name)
    print("docs in collection", self.collection.count())

  def add_documents(self, documents, embeddings):
    if(len(documents) != len(embeddings)):
      raise ValueError("number of documents does not match number of embeddings")

    #store -> ids, embedding, document, metadata
    ids = []
    all_metadata = []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate (zip(documents, embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata["doc_index"] = i
      metadata["content_length"] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

    self.collection.add(
        ids = ids,
        embeddings = embeddings_list,
        metadatas = all_metadata, # Changed from 'metadata' to 'metadatas'
        documents = documents_content
    )

    print("total documents added in vector store = ", len(documents_content))
    print("docs collection:", self.collection.count())

In [ ]:
vector_store = VectorStoreManager()

initialize the vector store with collection pdf_documents
docs in collection 0


In [ ]:
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

embedding shape: (390, 384)
total documents added in vector store =  390
docs collection: 390


Retrievel Pipeline

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class RAGRetriever:
  def __init__(self, embedding_manager, vector_store):
    self.embedding_manager = embedding_manager
    self.vector_store = vector_store

  def retrieve(self, query, top_k=5, score_threshold = 0.0):
    query_embedding = self.embedding_manager.generate_embeddings([query])[0]

    results = self.vector_store.collection.query(
        query_embeddings = [query_embedding],
        n_results = top_k
    )

    retrieved_docs = []
    if results["documents"] and results["documents"][0]:
      ids = results["ids"][0]
      documents = results["documents"][0]
      metadatas = results["metadatas"][0]
      distances = results["distances"][0]

      for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
        similarity_score = 1-distance # Note: This is a simplified similarity score based on L2 distance

        # Filter based on score_threshold
        # If score_threshold is 0.0, return all top_k results. Otherwise, apply the threshold.
        if score_threshold == 0.0 or similarity_score >= score_threshold:
          retrieved_docs.append(
              {
                  "id": doc_id,
                  "metadata": metadata,
                  "document": document,
                  "distance": distance,
                  "similarity_score": similarity_score,
                  "rank": i+1
              }

          )
    else:
      print("No documents found from vector store query.")

    print(f"Total documents retrieved after filtering: {len(retrieved_docs)}")
    return retrieved_docs

In [ ]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)


In [ ]:
rag_retriever.retrieve("What is RAG?")

embedding shape: (1, 384)
Total documents retrieved after filtering: 5


[{'id': 'doc_f81ca37f-8d8c-4d91-80c9-83fdc0dffed9',
  'metadata': {'page_label': '1',
   'doc_index': 157,
   'subject': '',
   'trapped': '/False',
   'total_pages': 21,
   'author': '',
   'source': 'data/research2.pdf',
   'page': 0,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'content_length': 288,
   'producer': 'pdfTeX-1.40.25',
   'keywords': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'creator': 'LaTeX with hyperref',
   'title': ''},
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'distance': 0.2314574122428894,
  'similarity_score': 0.7685425877571106,
  'rank': 1},
 {'id': 'doc_d9fea088-a7f0-

Groq

In [ ]:
from google.colab import userdata
API_Key_GROQ = userdata.get('GROQ_API_KEY')

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    groq_api_key = API_Key_GROQ,
    model = "qwen/qwen3-32b",
    temperature=0.1,
    max_tokens = 1024
)

In [ ]:
def generate_output(query, retriever, llm, top_k=3):
  results = retriever.retrieve(query, top_k)
  context = "\n".join([doc["document"] for doc in results]) if results else ""

  if not context:
    print("we found no relevant context for the given query")
  # context + query
  prompt = f"""use given context to generate answer for the query
  Context:{context}
  Query: {query} """

  response = llm.invoke([prompt])
  return response.content

In [ ]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

embedding shape: (1, 384)
Total documents retrieved after filtering: 3


In [ ]:
print(answer)

<think>
Okay, the user is asking, "What is encoder-decoder?" Let me start by recalling what I know about encoder-decoder models. They are a type of neural network architecture commonly used in tasks like machine translation, text summarization, and other sequence-to-sequence problems.

The encoder part processes the input data, converting it into a fixed-size context vector that captures the essential information. The decoder then uses this context vector to generate the output sequence. For example, in translation, the encoder processes the source language sentence, and the decoder generates the translated sentence in the target language.

Looking at the provided context, there's mention of models like BLIP-2 using frozen image encoders with LLMs for visual language tasks. Also, methods like "Visualize Before You Write" use image generation to guide text generation. These examples show encoders handling different modalities (images, text) and working with decoders to produce outputs. 

In [ ]:
!ls /content/*.ipynb

ls: cannot access '/content/*.ipynb': No such file or directory


In [ ]:
# Step 1: Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Find your notebook
!find /content/drive -name "*.ipynb" 2>/dev/null

In [42]:
import nbformat
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/RAG_Pipeline_AC.ipynb"  # <--- YOU NEED TO CHANGE THIS TO YOUR NOTEBOOK'S ACTUAL PATH/FILENAME


with open(notebook_path, "r") as f:
    nb = nbformat.read(f, as_version=4)


if "widgets" in nb.metadata:
    del nb.metadata["widgets"]


for cell in nb.cells:
    if "metadata" in cell and "widgets" in cell.metadata:
        del cell.metadata["widgets"]

with open(notebook_path, "w") as f:
    nbformat.write(nb, f)


from google.colab import files
files.download(notebook_path)

print("Done! Check your downloads folder.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Check your downloads folder.
